### Forest Area Calculation within the Biodiversity Exploritories

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely.ops import transform
from shapely.geometry import mapping
import warnings
warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)

import pystac
from pystac_client import Client # for STAC API access
import rioxarray # to work on remote and local rasters
import pyproj

In [3]:
# Read in Data and Shapefiles
alb = gpd.read_file("Boarders/Exploratorium_alb.gpkg")
hai = gpd.read_file("Boarders/Exploratorium_hai.gpkg")
sch = gpd.read_file("Boarders/Exploratorium_sch.gpkg")

In [4]:
# define the url for the STAC API
api_url = "https://earth-search.aws.element84.com/v1"
# Create a STAC client
client = Client.open(api_url)
# Collections
collections = client.get_collections()
# Print collection names
for collection in collections:
    print(collection)

<CollectionClient id=sentinel-2-pre-c1-l2a>
<CollectionClient id=cop-dem-glo-30>
<CollectionClient id=naip>
<CollectionClient id=cop-dem-glo-90>
<CollectionClient id=landsat-c2-l2>
<CollectionClient id=sentinel-2-l2a>
<CollectionClient id=sentinel-2-l1c>
<CollectionClient id=sentinel-2-c1-l2a>
<CollectionClient id=sentinel-1-grd>


With in Earth Search we can access sentinal, copernicus and some landsat. I will use Sentinal 2 because it has 10 m x 10 m resolution and flies over every 5 days

In [5]:
#help(client.search)

In [6]:
# Check and reproject to EPSG:4326 if necessary
if hai.crs != "EPSG:4326":
    hai = hai.to_crs("EPSG:4326")

# Now extract the bounding box from the reprojected geometry
minx, miny, maxx, maxy = hai.total_bounds
bbox_geom = {
    "type": "Polygon",
    "coordinates": [[
        [minx, miny],
        [minx, maxy],
        [maxx, maxy],
        [maxx, miny],
        [minx, miny]
    ]]
}

collection_sentinel_2_l2a = "sentinel-2-l2a"
search = client.search(
    collections=collection_sentinel_2_l2a,
    intersects=bbox_geom,
    datetime="2025-05-01/2025-05-31",
    query=['eo:cloud_cover<1']  # Limit the number of items returned
)

try:
    print(f'Number of items found: {search.matched()}')
except Exception as e:
    print("Error retrieving number of items found:", e)

Number of items found: 5


Only 5 images in the month of May match the criteria

In [7]:
# Save to JSON file
items = search.item_collection()
items.save_object("Hai_Sentinel.JSON")

In [8]:
assets = items[0].assets
# Print asset information
for key, asset in assets.items():
    print(f"{key}: {asset.title}")



aot: Aerosol optical thickness (AOT)
blue: Blue - 10m
cloud: Cloud Probabilities
coastal: Coastal - 60m
granule_metadata: None
green: Green - 10m
nir: NIR 1 - 10m
nir08: NIR 2 - 20m
nir09: NIR 3 - 60m
product_metadata: None
red: Red - 10m
rededge1: Red Edge 1 - 20m
rededge2: Red Edge 2 - 20m
rededge3: Red Edge 3 - 20m
scl: Scene classification map (SCL)
snow: Snow Probabilities
swir16: SWIR 1.6μm - 20m
swir22: SWIR 2.2μm - 20m
tileinfo_metadata: None
visual: True color image
wvp: Water Vapour (WVP)
thumbnail: Thumbnail of preview image
aot-jp2: Aerosol optical thickness (AOT)
blue-jp2: Blue - 10m
coastal-jp2: Coastal - 60m
green-jp2: Green - 10m
nir-jp2: NIR 1 - 10m
nir08-jp2: NIR 2 - 20m
nir09-jp2: NIR 3 - 60m
red-jp2: Red - 10m
rededge1-jp2: Red Edge 1 - 20m
rededge2-jp2: Red Edge 2 - 20m
rededge3-jp2: Red Edge 3 - 20m
scl-jp2: Scene classification map (SCL)
swir16-jp2: SWIR 1.6μm - 20m
swir22-jp2: SWIR 2.2μm - 20m
visual-jp2: True color image
wvp-jp2: Water Vapour (WVP)


In [9]:
#help(rioxarray.open_rasterio)

In [10]:
# Lets use the green band first
nir_href = assets["nir"].href # for the red band
nir_band =rioxarray.open_rasterio(nir_href,masked=True)
print(nir_band)

red_href = assets["red"].href # for the red band
red_band =rioxarray.open_rasterio(red_href,masked=True)
print(red_band)

<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 482MB
[120560400 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 5.7e+06 5.7e+06 5.7e+06 ... 5.59e+06 5.59e+06
    spatial_ref  int64 8B 0
Attributes:
    OVR_RESAMPLING_ALG:  AVERAGE
    AREA_OR_POINT:       Area
    scale_factor:        1.0
    add_offset:          0.0
<xarray.DataArray (band: 1, y: 10980, x: 10980)> Size: 482MB
[120560400 values with dtype=float32]
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
  * y            (y) float64 88kB 5.7e+06 5.7e+06 5.7e+06 ... 5.59e+06 5.59e+06
    spatial_ref  int64 8B 0
Attributes:
    OVR_RESAMPLING_ALG:  AVERAGE
    AREA_OR_POINT:       Area
    scale_factor:        1.0
    add_offset:          0.0


In [11]:
nir_band.rio.to_raster("nir_band.tif")
red_band.rio.to_raster("red_band.tif")

RasterioIOError: Read failed. See previous exception for details.

You are already using `masked=True` in `rioxarray.open_rasterio`, which will automatically mask nodata pixels.  
If you still encounter issues with a single pixel, it may be due to:
- The presence of NaN or masked values at the edges.
- The bounding box or geometry slightly exceeding the raster extent.
- Data alignment or CRS mismatch.

**Tips:**
- You can further mask out NaN values using `.where(~nir_band.isnull())`.
- Always check the raster's `.rio.bounds()` and compare with your geometry.
- Visualize the mask with `nir_band.plot()` to inspect problematic areas.

Example:
```python
nir_band = rioxarray.open_rasterio(nir_href, masked=True)
nir_band = nir_band.where(~nir_band.isnull())
```
If the issue persists, inspect the raster metadata and your geometry for subtle mismatches.

If you see errors like  
```
CPLE_AppDefinedError: TIFFFillTile:Read error at row 4294967295, col 4294967295, tile 0; got 0 bytes, expected ...
CPLE_AppDefinedError: TIFFReadEncodedTile() failed.
```
this usually means the TIFF file is corrupted or incomplete, or there was a problem during download or reading.

**How to address:**
- Try re-downloading the file or using a different asset link.
- Check your internet connection if reading from a remote source.
- If possible, open the file with another tool (e.g., QGIS) to verify its integrity.
- If the error is only for a single band or file, try skipping that file and using another.

If you are reading from S3 or a remote URL, sometimes the error is temporary and a retry may succeed.
</markdown>
````